In [1]:
%pip install crewai langchain langchain-openai langchain-community langchain-tavily tavily-python pydantic

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install litellm
%pip install -U crewai

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import requests
import litellm
from crewai.llm import LLM
from crewai import Agent, Task, Crew
from crewai.tools import BaseTool
from dotenv import load_dotenv

loaded = load_dotenv()

# Configure LiteLLM
litellm.drop_params = True
# Monkey patch litellm.completion to handle parameter mapping
original_completion = litellm.completion

def patched_completion(*args, **kwargs):
    # If max_tokens is present and max_completion_tokens is not, map it
    if 'max_tokens' in kwargs and 'max_completion_tokens' not in kwargs:
        kwargs['max_completion_tokens'] = kwargs.pop('max_tokens')
    return original_completion(*args, **kwargs)

# Apply the patch
litellm.completion = patched_completion

In [4]:
# Tavily API Key
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

In [ ]:
# ---- Custom CrewAI Tool for Web Search ----
class TavilySearchTool(BaseTool):
    name: str = "web_search"
    description: str = "Search the web for recent information."

    def _run(self, query: str):
        url = "https://api.tavily.com/search"
        payload = {
            "api_key": TAVILY_API_KEY,
            "query": query,
            "max_results": 3
        }

        response = requests.post(url, json=payload, timeout=30)
        response.raise_for_status()
        data = response.json()

        results = []
        for r in data.get("results", []):
            title = r.get("title", "No title")
            link = r.get("url", "No URL")
            results.append(f"{title} - {link}")

        return "\n".join(results) if results else "No web results found."


search_tool = TavilySearchTool()

# ---- Azure LLM - FIXED ----
# Using max_tokens (not max_completion_tokens) with the monkey patch
llm = LLM(
    model=f"azure/{os.getenv('AZURE_OPENAI_CHAT_DEPLOYMENT')}",
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    base_url=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-12-01-preview"),
    is_litellm=True,
    temperature=0,
    max_tokens=3500  # This will be converted to max_completion_tokens
)

In [6]:
# --------------------------------------------------
# Agents
# --------------------------------------------------
researcher = Agent(
    role="AI Researcher",
    goal="Find the latest advancements in AI for healthcare",
    backstory=(
        "You are an expert in artificial intelligence and stay updated "
        "with the latest research trends in healthcare."
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm,
    max_iter=2,
    tools=[search_tool]
)

writer = Agent(
    role="Technical Writer",
    goal="Summarize research into an executive report",
    backstory=(
        "You are an experienced technical writer with expertise in "
        "summarizing healthcare research for executives."
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm
)

In [7]:
# --------------------------------------------------
# SERIAL EXECUTION
# --------------------------------------------------
task_research = Task(
    description=(
        "Use the web search results to explain the top 3 recent advancements in AI for healthcare in 3-4 sentences."
        "Do not call tools again after getting results."
    ),
    expected_output=(
        "Detailed notes on three advancements, with names and explanations."
    ),
    agent=researcher
)

task_write = Task(
    description=(
        "Write a short executive summary using the research notes provided by the AI Researcher. "
        "Limit the answer to about 100 words."
    ),
    expected_output=(
        "An executive summary report of the top 3 AI advancements in healthcare."
    ),
    agent=writer,
    context=[task_research]
)

print("\n=== SERIAL EXECUTION ===")

crew_serial = Crew(
    agents=[researcher, writer],
    tasks=[task_research, task_write],
    verbose=True
)

serial_result = await crew_serial.kickoff_async()

print("\n[Serial Result]:\n")
try:
    print(serial_result.raw)
except AttributeError:
    print(serial_result)


=== SERIAL EXECUTION ===


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 661bd4f0-8869-49d5-88d5-46d5ee5290be                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Use the web search results to explain the top 3 recent advancements in AI for healthcare in 3-4          │
│  sentences.Do not call tools again after getting results.                                                       │
│  ID: f5237975-cd53-49f9-ad99-163bb28e59b5                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Researcher                                                                                           │
│                                                                                                                 │
│  Task: Use the web search results to explain the top 3 recent advancements in AI for healthcare in 3-4          │
│  sentences.Do not call tools again after getting results.                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: web_search                                                                                               │
│  Args: {"query": "recent top advancements in AI for healthcare 2023 2024 2025 large language models Med-PaLM 2  │
│  clinical LLMs AlphaFold protein folding drug discovery generative models medical imaging diffusi...            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Executive Summary: Three AI advances are reshaping healthcare. 1) Clinical LLMs (e.g., Med‑PaLM2, specialized  │
│  GPT/LLaMA derivatives) enhance clinical Q&A, discharge summaries, charting, decision support and EHR           │
│  conversational access, cutting documentation time while requiring rigorous validation to mitigate              │
│  hallucination and safety risks. 2) AI‑driven protein folding and molecular design (AlphaFold2, RoseTTAFold,    │
│  ProteinMPNN, generative/diffusion models) enable accurate structure prediction, in silico target               │
│  identification and faster lead prioritization, shortening preclinical cycles despite limits in dynamics,       │
│  complexes and biological context. 3) Generative and multimodal imaging/foundation models improve               │
│  reconstruction, segmentation, report automation and cross‑modal interpretation, with clinical adoption         │
│  growing but ongoing generalization and bias evaluation needed.                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Write a short executive summary using the research notes provided by the AI Researcher. Limit the        │
│  answer to about 100 words.                                                                                     │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 661bd4f0-8869-49d5-88d5-46d5ee5290be                                                                       │
│  Final Output: Executive Summary: Three AI advances are reshaping healthcare. 1) Clinical LLMs (e.g.,           │
│  Med‑PaLM2, specialized GPT/LLaMA derivatives) enhance clinical Q&A, discharge summaries, charting, decision    │
│  support and EHR conversational access, cutting documentation time while requiring rigorous validation to       │
│  mitigate hallucination and safety risks. 2) AI‑driven protein folding and molecular design (AlphaFold2,        │
│  RoseTTAFold, ProteinMPNN, generative/diffusion models) enable accurate structure prediction, in silico target  │
│  identification and faster lead prioritization, shortening preclinical cycles despite limits in dynamics,       │
│  complexes and biological context. 3) Generative and multimodal imaging/foundation models improve               │
│  reconstruction, segmentation, report automation and cross‑modal interpretation, with clinical adoption         │
│  growing but ongoing generalization and bias evaluation needed.                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


[Serial Result]:

Executive Summary: Three AI advances are reshaping healthcare. 1) Clinical LLMs (e.g., Med‑PaLM2, specialized GPT/LLaMA derivatives) enhance clinical Q&A, discharge summaries, charting, decision support and EHR conversational access, cutting documentation time while requiring rigorous validation to mitigate hallucination and safety risks. 2) AI‑driven protein folding and molecular design (AlphaFold2, RoseTTAFold, ProteinMPNN, generative/diffusion models) enable accurate structure prediction, in silico target identification and faster lead prioritization, shortening preclinical cycles despite limits in dynamics, complexes and biological context. 3) Generative and multimodal imaging/foundation models improve reconstruction, segmentation, report automation and cross‑modal interpretation, with clinical adoption growing but ongoing generalization and bias evaluation needed.


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [8]:
# --------------------------------------------------
# PARALLEL EXECUTION
# --------------------------------------------------
task_parallel_1 = Task(
    description=(
        "Use web search to list 5 AI companies focusing on drug discovery. "
        "For each company, give one short line about what they specialize in."
    ),
    expected_output="Company names and what they specialize in.",
    async_execution=True,
    agent=researcher
)

task_parallel_2 = Task(
    description=(
        "Write a short report on how AI is transforming patient diagnostics. "
        "Limit the answer to about 100 words."
    ),
    expected_output="A short report with examples and explanation.",
    agent=writer
)

print("\n=== PARALLEL EXECUTION ===")

crew_parallel = Crew(
    agents=[researcher, writer],
    tasks=[task_parallel_1, task_parallel_2],
    verbose=True
)

parallel_result = await crew_parallel.kickoff_async()

print("\n[Parallel Result]:\n")
try:
    print(parallel_result.raw)
except AttributeError:
    print(parallel_result)


=== PARALLEL EXECUTION ===


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 14c328f7-0f13-4581-8f7b-3d38cd9dd0e9                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Use web search to list 5 AI companies focusing on drug discovery. For each company, give one short line  │
│  about what they specialize in.                                                                                 │
│  ID: bd4c7e5c-a471-4db3-9b57-139926989994                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Researcher                                                                                           │
│                                                                                                                 │
│  Task: Use web search to list 5 AI companies focusing on drug discovery. For each company, give one short line  │
│  about what they specialize in.                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#2) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: web_search                                                                                               │
│  Args: {"query": "Atomwise company specializes AI drug discovery what they do"}                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  AI is transforming patient diagnostics by enabling faster, more accurate detection through pattern             │
│  recognition, multimodal data integration, and biomarker discovery. Example: Recursion combines                 │
│  high-throughput cellular imaging and machine learning to identify disease phenotypes and early cellular        │
│  signals; Insilico Medicine uses generative AI for biomarker discovery that supports early diagnostic assays;   │
│  BenevolentAI's knowledge-graph integrates heterogeneous biomedical data to suggest diagnostic markers and      │
│  differential diagnoses. Atomwise and Exscientia’s molecular-prediction platforms accelerate identification of  │
│  protein interactions and probes used in diagnostic tests. Together these approaches reduce time-to-diagnosis,  │
│  increase sensitivity/specificity, and enable personalized diagnostic pathways for earlier, targeted            │
│  treatment.                                                                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Write a short report on how AI is transforming patient diagnostics. Limit the answer to about 100        │
│  words.                                                                                                         │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 14c328f7-0f13-4581-8f7b-3d38cd9dd0e9                                                                       │
│  Final Output: AI is transforming patient diagnostics by enabling faster, more accurate detection through       │
│  pattern recognition, multimodal data integration, and biomarker discovery. Example: Recursion combines         │
│  high-throughput cellular imaging and machine learning to identify disease phenotypes and early cellular        │
│  signals; Insilico Medicine uses generative AI for biomarker discovery that supports early diagnostic assays;   │
│  BenevolentAI's knowledge-graph integrates heterogeneous biomedical data to suggest diagnostic markers and      │
│  differential diagnoses. Atomwise and Exscientia’s molecular-prediction platforms accelerate identification of  │
│  protein interactions and probes used in diagnostic tests. Together these approaches reduce time-to-diagnosis,  │
│  increase sensitivity/specificity, and enable personalized diagnostic pathways for earlier, targeted            │
│  treatment.                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


[Parallel Result]:

AI is transforming patient diagnostics by enabling faster, more accurate detection through pattern recognition, multimodal data integration, and biomarker discovery. Example: Recursion combines high-throughput cellular imaging and machine learning to identify disease phenotypes and early cellular signals; Insilico Medicine uses generative AI for biomarker discovery that supports early diagnostic assays; BenevolentAI's knowledge-graph integrates heterogeneous biomedical data to suggest diagnostic markers and differential diagnoses. Atomwise and Exscientia’s molecular-prediction platforms accelerate identification of protein interactions and probes used in diagnostic tests. Together these approaches reduce time-to-diagnosis, increase sensitivity/specificity, and enable personalized diagnostic pathways for earlier, targeted treatment.


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯